# 07 - E5 Evaluacion de seleccion de slices sin mascara

**Objetivo:** evaluar el impacto de reemplazar la seleccion de slice basada en mascara por estrategias que no dependan de la mascara.

Este notebook carga el modelo binario mejorado, usa los mismos casos holdout del notebook 06 y compara estrategias de seleccion de slice:

- referencia con mascara: `mask_max_area`;
- estrategias sin mascara: `central_slice`, `center_window_best_prediction`, `all_slices_best_prediction`;
- estrategia adicional: `top_k_prediction_mean`.

**Fuera de alcance:** reentrenamiento, cambio de arquitectura, multiclase, axial, nnU-Net e integracion backend.

## 1. Instalacion e importacion de dependencias

In [ ]:
!pip -q install SimpleITK scikit-image tqdm

In [ ]:
from pathlib import Path
import json
import random
import warnings

import SimpleITK as sitk
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from skimage.transform import resize
from tqdm.auto import tqdm

import torch
import torch.nn as nn

warnings.filterwarnings("ignore")

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("PyTorch:", torch.__version__)
print("SimpleITK:", sitk.Version())
print("CUDA disponible:", torch.cuda.is_available())
print("Dispositivo:", DEVICE)

## 2. Montaje de Drive y definicion de rutas

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
DATASET_ROOT = Path("/content/drive/MyDrive/PFI_MVP/data/SPIDER")
PREPROCESS_ROOT = Path("/content/drive/MyDrive/PFI_MVP/results/E4_preprocesamiento")
IMPROVED_ROOT = Path("/content/drive/MyDrive/PFI_MVP/results/E5_baseline_mejorado_binario")
HOLDOUT_ROOT = Path("/content/drive/MyDrive/PFI_MVP/results/E5_holdout_validacion")
SLICE_EVAL_ROOT = Path("/content/drive/MyDrive/PFI_MVP/results/E5_slice_selection_eval")
FIGURES_ROOT = Path("/content/drive/MyDrive/PFI_MVP/figures")
DOCS_ROOT = Path("/content/drive/MyDrive/PFI_MVP/docs")

for path in [SLICE_EVAL_ROOT, FIGURES_ROOT, DOCS_ROOT]:
    path.mkdir(parents=True, exist_ok=True)

MODEL_PATH = IMPROVED_ROOT / "E5_improved_unet2d_binary_best.pt"
HOLDOUT_SELECTED_CSV = HOLDOUT_ROOT / "E5_holdout_selected_cases.csv"
HOLDOUT_REPORT_JSON = HOLDOUT_ROOT / "E5_holdout_validation_report.json"

print("MODEL_PATH:", MODEL_PATH)
print("HOLDOUT_SELECTED_CSV:", HOLDOUT_SELECTED_CSV)
print("HOLDOUT_REPORT_JSON:", HOLDOUT_REPORT_JSON)
print("SLICE_EVAL_ROOT:", SLICE_EVAL_ROOT)

## 3. Carga del modelo mejorado

In [ ]:
class DoubleConv(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, 3, padding=1),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_channels, out_channels, 3, padding=1),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
        )

    def forward(self, x):
        return self.block(x)


class SimpleUNet2D(nn.Module):
    def __init__(self, in_channels=1, out_channels=1, base_channels=16):
        super().__init__()
        self.enc1 = DoubleConv(in_channels, base_channels)
        self.enc2 = DoubleConv(base_channels, base_channels * 2)
        self.enc3 = DoubleConv(base_channels * 2, base_channels * 4)
        self.pool = nn.MaxPool2d(2)
        self.bottleneck = DoubleConv(base_channels * 4, base_channels * 8)
        self.up3 = nn.ConvTranspose2d(base_channels * 8, base_channels * 4, 2, stride=2)
        self.dec3 = DoubleConv(base_channels * 8, base_channels * 4)
        self.up2 = nn.ConvTranspose2d(base_channels * 4, base_channels * 2, 2, stride=2)
        self.dec2 = DoubleConv(base_channels * 4, base_channels * 2)
        self.up1 = nn.ConvTranspose2d(base_channels * 2, base_channels, 2, stride=2)
        self.dec1 = DoubleConv(base_channels * 2, base_channels)
        self.out = nn.Conv2d(base_channels, out_channels, 1)

    def forward(self, x):
        e1 = self.enc1(x)
        e2 = self.enc2(self.pool(e1))
        e3 = self.enc3(self.pool(e2))
        b = self.bottleneck(self.pool(e3))
        d3 = self.dec3(torch.cat([self.up3(b), e3], dim=1))
        d2 = self.dec2(torch.cat([self.up2(d3), e2], dim=1))
        d1 = self.dec1(torch.cat([self.up1(d2), e1], dim=1))
        return self.out(d1)


if not MODEL_PATH.exists():
    raise FileNotFoundError(f"No existe el modelo mejorado: {MODEL_PATH}")

checkpoint = torch.load(MODEL_PATH, map_location=DEVICE)
BASE_CHANNELS = int(checkpoint.get("base_channels", 16))
TARGET_SIZE = tuple(checkpoint.get("target_size", (256, 256)))
SAGITTAL_AXIS = int(checkpoint.get("sagittal_axis", 2))

model = SimpleUNet2D(base_channels=BASE_CHANNELS).to(DEVICE)
model.load_state_dict(checkpoint["model_state_dict"])
model.eval()

print("Modelo cargado:", MODEL_PATH)
print("BASE_CHANNELS:", BASE_CHANNELS)
print("TARGET_SIZE:", TARGET_SIZE)
print("SAGITTAL_AXIS:", SAGITTAL_AXIS)

## 4. Carga del holdout

In [ ]:
N_EVAL = 40
THRESHOLD = 0.5
CENTER_WINDOW_RADIUS = 3
TOP_K = 3
INFERENCE_BATCH_SIZE = 16

if not HOLDOUT_SELECTED_CSV.exists():
    raise FileNotFoundError(f"No existe holdout seleccionado: {HOLDOUT_SELECTED_CSV}")

holdout_df = pd.read_csv(HOLDOUT_SELECTED_CSV)
if "case_id" not in holdout_df.columns:
    for candidate in ["file_name", "image_path", "source_image_path"]:
        if candidate in holdout_df.columns:
            holdout_df["case_id"] = holdout_df[candidate].apply(lambda x: Path(str(x)).stem)
            break
holdout_df["case_id"] = holdout_df["case_id"].astype(str)

eval_df = holdout_df.head(min(N_EVAL, len(holdout_df))).copy().reset_index(drop=True)

print("Casos holdout disponibles:", len(holdout_df))
print("Casos a evaluar:", len(eval_df))
display(eval_df.head())

## 5. Funciones de preprocesamiento

La mascara se conserva solo para evaluacion. Las estrategias sin mascara no la usan para elegir el slice.

In [ ]:
def resolve_path(value):
    return Path(str(value))


def get_case_paths(row):
    image_candidates = ["image_path", "source_image_path", "image", "img_path"]
    mask_candidates = ["mask_path", "source_mask_path", "mask", "seg_path"]
    image_path = None
    mask_path = None
    for column in image_candidates:
        if column in row and pd.notna(row[column]):
            image_path = resolve_path(row[column])
            break
    for column in mask_candidates:
        if column in row and pd.notna(row[column]):
            mask_path = resolve_path(row[column])
            break
    if image_path is None or mask_path is None:
        raise ValueError("No se encontraron columnas de path para imagen/mascara.")
    return image_path, mask_path


def read_mha(path: Path):
    itk_image = sitk.ReadImage(str(path))
    array = sitk.GetArrayFromImage(itk_image)
    return itk_image, array


def robust_percentile_normalize(image_array, p_low=1, p_high=99, eps=1e-8):
    image_float = image_array.astype(np.float32)
    low, high = np.percentile(image_float, [p_low, p_high])
    clipped = np.clip(image_float, low, high)
    if np.isclose(high, low):
        return np.zeros_like(clipped, dtype=np.float32)
    return ((clipped - low) / (high - low + eps)).astype(np.float32)


def take_slice(array, axis, index):
    return np.take(array, indices=index, axis=axis)


def resize_slice(array_2d, target_size=TARGET_SIZE, order=1):
    return resize(
        array_2d,
        output_shape=target_size,
        order=order,
        preserve_range=True,
        anti_aliasing=(order != 0),
    ).astype(np.float32)


def preprocess_volume(row):
    image_path, mask_path = get_case_paths(row)
    itk_image, image = read_mha(image_path)
    itk_mask, mask = read_mha(mask_path)

    if image.shape != mask.shape:
        raise ValueError(f"Shape incompatible: image={image.shape}, mask={mask.shape}")

    image_norm = robust_percentile_normalize(image)
    mask_bin = (mask > 0).astype(np.float32)
    return {
        "case_id": str(row["case_id"]),
        "image": image_norm,
        "mask": mask_bin,
        "source_image_path": str(image_path),
        "source_mask_path": str(mask_path),
    }


def prepare_slice(volume, slice_index):
    image_slice = take_slice(volume["image"], SAGITTAL_AXIS, slice_index)
    mask_slice = take_slice(volume["mask"], SAGITTAL_AXIS, slice_index)
    image_slice = resize_slice(image_slice, order=1)
    mask_slice = resize_slice(mask_slice, order=0)
    mask_slice = (mask_slice > 0.5).astype(np.float32)
    return image_slice.astype(np.float32), mask_slice.astype(np.float32)


def dice_iou_numpy(pred_mask, gt_mask, eps=1e-7):
    pred = pred_mask.astype(bool)
    gt = gt_mask.astype(bool)
    intersection = np.logical_and(pred, gt).sum()
    pred_sum = pred.sum()
    gt_sum = gt.sum()
    union = np.logical_or(pred, gt).sum()
    dice = (2.0 * intersection + eps) / (pred_sum + gt_sum + eps)
    iou = (intersection + eps) / (union + eps)
    return float(dice), float(iou)

## 6. Estrategias de seleccion de slice

In [ ]:
def mask_max_area_slice(volume):
    reduce_axes = tuple(ax for ax in range(volume["mask"].ndim) if ax != SAGITTAL_AXIS)
    area_by_slice = np.sum(volume["mask"] > 0, axis=reduce_axes)
    return int(np.argmax(area_by_slice))


def central_slice(volume):
    return int(volume["image"].shape[SAGITTAL_AXIS] // 2)


def candidate_center_window(volume, radius=CENTER_WINDOW_RADIUS):
    center = central_slice(volume)
    start = max(0, center - radius)
    end = min(volume["image"].shape[SAGITTAL_AXIS] - 1, center + radius)
    return list(range(start, end + 1))


def candidate_all_slices(volume):
    return list(range(volume["image"].shape[SAGITTAL_AXIS]))


def infer_slices(volume, slice_indices):
    image_slices = []
    mask_slices = []
    for slice_index in slice_indices:
        image_slice, mask_slice = prepare_slice(volume, slice_index)
        image_slices.append(image_slice)
        mask_slices.append(mask_slice)

    probs = []
    model.eval()
    with torch.no_grad():
        for start in range(0, len(image_slices), INFERENCE_BATCH_SIZE):
            batch_images = np.stack(image_slices[start:start + INFERENCE_BATCH_SIZE], axis=0)
            batch_tensor = torch.from_numpy(batch_images[:, None, :, :]).float().to(DEVICE)
            logits = model(batch_tensor)
            batch_probs = torch.sigmoid(logits).detach().cpu().numpy()[:, 0]
            probs.extend(list(batch_probs))

    records = []
    for slice_index, image_slice, mask_slice, prob_slice in zip(slice_indices, image_slices, mask_slices, probs):
        pred_slice = (prob_slice >= THRESHOLD).astype(np.float32)
        dice, iou = dice_iou_numpy(pred_slice, mask_slice)
        records.append({
            "slice_index": int(slice_index),
            "image": image_slice,
            "mask": mask_slice,
            "prob": prob_slice,
            "pred": pred_slice,
            "pred_foreground_ratio": float(pred_slice.mean()),
            "gt_foreground_ratio": float(mask_slice.mean()),
            "dice": dice,
            "iou": iou,
        })
    return records


STRATEGIES = [
    "mask_max_area",
    "central_slice",
    "center_window_best_prediction",
    "all_slices_best_prediction",
    "top_k_prediction_mean",
]

## 7. Inferencia por estrategia

In [ ]:
def evaluate_strategy(volume, strategy):
    if strategy == "mask_max_area":
        selected_idx = mask_max_area_slice(volume)
        records = infer_slices(volume, [selected_idx])
        record = records[0]
        return {
            "selected_slice_index": selected_idx,
            "uses_mask_for_selection": True,
            "dice": record["dice"],
            "iou": record["iou"],
            "gt_foreground_ratio": record["gt_foreground_ratio"],
            "pred_foreground_ratio": record["pred_foreground_ratio"],
            "record": record,
        }

    if strategy == "central_slice":
        selected_idx = central_slice(volume)
        records = infer_slices(volume, [selected_idx])
        record = records[0]
        return {
            "selected_slice_index": selected_idx,
            "uses_mask_for_selection": False,
            "dice": record["dice"],
            "iou": record["iou"],
            "gt_foreground_ratio": record["gt_foreground_ratio"],
            "pred_foreground_ratio": record["pred_foreground_ratio"],
            "record": record,
        }

    if strategy == "center_window_best_prediction":
        records = infer_slices(volume, candidate_center_window(volume))
        record = max(records, key=lambda x: x["pred_foreground_ratio"])
        return {
            "selected_slice_index": record["slice_index"],
            "uses_mask_for_selection": False,
            "dice": record["dice"],
            "iou": record["iou"],
            "gt_foreground_ratio": record["gt_foreground_ratio"],
            "pred_foreground_ratio": record["pred_foreground_ratio"],
            "record": record,
        }

    if strategy == "all_slices_best_prediction":
        records = infer_slices(volume, candidate_all_slices(volume))
        record = max(records, key=lambda x: x["pred_foreground_ratio"])
        return {
            "selected_slice_index": record["slice_index"],
            "uses_mask_for_selection": False,
            "dice": record["dice"],
            "iou": record["iou"],
            "gt_foreground_ratio": record["gt_foreground_ratio"],
            "pred_foreground_ratio": record["pred_foreground_ratio"],
            "record": record,
        }

    if strategy == "top_k_prediction_mean":
        records = infer_slices(volume, candidate_all_slices(volume))
        top_records = sorted(records, key=lambda x: x["pred_foreground_ratio"], reverse=True)[:TOP_K]
        best_record = top_records[0]
        return {
            "selected_slice_index": best_record["slice_index"],
            "uses_mask_for_selection": False,
            "dice": float(np.mean([r["dice"] for r in top_records])),
            "iou": float(np.mean([r["iou"] for r in top_records])),
            "gt_foreground_ratio": float(np.mean([r["gt_foreground_ratio"] for r in top_records])),
            "pred_foreground_ratio": float(np.mean([r["pred_foreground_ratio"] for r in top_records])),
            "record": best_record,
        }

    raise ValueError(f"Estrategia no soportada: {strategy}")


metrics_rows = []
visualization_cache = []

for _, row in tqdm(eval_df.iterrows(), total=len(eval_df), desc="Evaluando casos"):
    volume = preprocess_volume(row)
    case_results = {}

    for strategy in STRATEGIES:
        result = evaluate_strategy(volume, strategy)
        case_results[strategy] = result
        metrics_rows.append({
            "case_id": volume["case_id"],
            "strategy": strategy,
            "uses_mask_for_selection": result["uses_mask_for_selection"],
            "selected_slice_index": result["selected_slice_index"],
            "dice": result["dice"],
            "iou": result["iou"],
            "gt_foreground_ratio": result["gt_foreground_ratio"],
            "pred_foreground_ratio": result["pred_foreground_ratio"],
        })

    if len(visualization_cache) < 3:
        visualization_cache.append({
            "case_id": volume["case_id"],
            "results": case_results,
        })

metrics_df = pd.DataFrame(metrics_rows)
metrics_csv_path = SLICE_EVAL_ROOT / "E5_slice_selection_metrics_by_case.csv"
metrics_df.to_csv(metrics_csv_path, index=False)

print("CSV metricas por caso:", metrics_csv_path)
display(metrics_df.head())

## 8. Resumen por estrategia

In [ ]:
summary_df = (
    metrics_df.groupby(["strategy", "uses_mask_for_selection"], as_index=False)
    .agg(
        dice_mean=("dice", "mean"),
        dice_std=("dice", "std"),
        iou_mean=("iou", "mean"),
        iou_std=("iou", "std"),
        pred_foreground_ratio_mean=("pred_foreground_ratio", "mean"),
        gt_foreground_ratio_mean=("gt_foreground_ratio", "mean"),
        n_cases=("case_id", "nunique"),
    )
    .sort_values("dice_mean", ascending=False)
)

summary_csv_path = SLICE_EVAL_ROOT / "E5_slice_selection_summary.csv"
summary_df.to_csv(summary_csv_path, index=False)

print("CSV resumen:", summary_csv_path)
display(summary_df)

## 9. Comparacion contra referencia con mascara

In [ ]:
reference_df = metrics_df[metrics_df["strategy"] == "mask_max_area"][
    ["case_id", "dice", "iou"]
].rename(columns={"dice": "reference_dice", "iou": "reference_iou"})

comparison_df = metrics_df.merge(reference_df, on="case_id", how="left")
comparison_df["dice_delta_vs_mask_reference"] = comparison_df["dice"] - comparison_df["reference_dice"]
comparison_df["iou_delta_vs_mask_reference"] = comparison_df["iou"] - comparison_df["reference_iou"]

comparison_summary_df = (
    comparison_df.groupby(["strategy", "uses_mask_for_selection"], as_index=False)
    .agg(
        dice_mean=("dice", "mean"),
        reference_dice_mean=("reference_dice", "mean"),
        dice_delta_mean=("dice_delta_vs_mask_reference", "mean"),
        iou_mean=("iou", "mean"),
        reference_iou_mean=("reference_iou", "mean"),
        iou_delta_mean=("iou_delta_vs_mask_reference", "mean"),
        n_cases=("case_id", "nunique"),
    )
)

comparison_csv_path = SLICE_EVAL_ROOT / "E5_slice_selection_vs_mask_reference.csv"
comparison_summary_df.to_csv(comparison_csv_path, index=False)

print("CSV comparacion contra referencia:", comparison_csv_path)
display(comparison_summary_df)

## 10. Visualizaciones comparativas

In [ ]:
def export_strategy_figure(cache_item, index):
    strategies_to_plot = [
        "mask_max_area",
        "central_slice",
        "center_window_best_prediction",
        "all_slices_best_prediction",
    ]

    fig, axes = plt.subplots(2, len(strategies_to_plot), figsize=(20, 8))

    for col, strategy in enumerate(strategies_to_plot):
        result = cache_item["results"][strategy]
        record = result["record"]
        axes[0, col].imshow(record["image"], cmap="gray", vmin=0, vmax=1)
        axes[0, col].imshow(np.ma.masked_where(record["mask"] == 0, record["mask"]), cmap="Greens", alpha=0.35)
        axes[0, col].set_title(f"{strategy}\nslice {result['selected_slice_index']}")
        axes[0, col].axis("off")

        axes[1, col].imshow(record["image"], cmap="gray", vmin=0, vmax=1)
        axes[1, col].imshow(np.ma.masked_where(record["pred"] == 0, record["pred"]), cmap="Reds", alpha=0.40)
        axes[1, col].set_title(f"Dice {result['dice']:.3f} / IoU {result['iou']:.3f}")
        axes[1, col].axis("off")

    fig.suptitle(f"Comparacion estrategias de slice - {cache_item['case_id']}")
    fig.tight_layout()

    path = FIGURES_ROOT / f"E5_slice_strategy_example_{index:02d}.png"
    fig.savefig(path, dpi=150, bbox_inches="tight")
    plt.show()
    return path


figure_paths = []
for index, item in enumerate(visualization_cache, start=1):
    figure_paths.append(export_strategy_figure(item, index))

print("Figuras exportadas:")
for path in figure_paths:
    print(path)

## 11. Reporte JSON

In [ ]:
reference_row = summary_df[summary_df["strategy"] == "mask_max_area"].iloc[0]
no_mask_summary_df = summary_df[summary_df["uses_mask_for_selection"] == False].copy()
best_no_mask_row = no_mask_summary_df.loc[no_mask_summary_df["dice_mean"].idxmax()]

dice_drop_absolute = float(reference_row["dice_mean"] - best_no_mask_row["dice_mean"])
dice_drop_relative = float(dice_drop_absolute / max(reference_row["dice_mean"], 1e-7))
iou_drop_absolute = float(reference_row["iou_mean"] - best_no_mask_row["iou_mean"])

if dice_drop_absolute <= 0.05:
    recommendation = "La mejor estrategia sin mascara queda cerca de la referencia; es viable avanzar con seleccion sin mascara y ampliar evaluacion."
elif best_no_mask_row["strategy"] in ["all_slices_best_prediction", "top_k_prediction_mean"]:
    recommendation = "La seleccion sin mascara funciona mejor al explorar multiples slices; conviene avanzar hacia inferencia multi-slice."
else:
    recommendation = "La caida frente a la referencia es relevante; conviene evaluar inferencia multi-slice o un selector de slice dedicado."

report = {
    "evaluated_cases": int(metrics_df["case_id"].nunique()),
    "strategies": STRATEGIES,
    "best_no_mask_strategy_by_dice": str(best_no_mask_row["strategy"]),
    "reference_mask_max_area_dice": float(reference_row["dice_mean"]),
    "best_no_mask_dice": float(best_no_mask_row["dice_mean"]),
    "dice_drop_absolute": dice_drop_absolute,
    "dice_drop_relative": dice_drop_relative,
    "reference_mask_max_area_iou": float(reference_row["iou_mean"]),
    "best_no_mask_iou": float(best_no_mask_row["iou_mean"]),
    "iou_drop_absolute": iou_drop_absolute,
    "recommendation": recommendation,
    "exports": {
        "metrics_by_case": str(metrics_csv_path),
        "summary": str(summary_csv_path),
        "comparison_vs_mask_reference": str(comparison_csv_path),
        "figures": [str(path) for path in figure_paths],
    },
}

report_json_path = SLICE_EVAL_ROOT / "E5_slice_selection_report.json"
report_json_path.write_text(json.dumps(report, indent=2), encoding="utf-8")

print("Reporte JSON:", report_json_path)
print(json.dumps(report, indent=2))

## 12. Conclusion tecnica Markdown

In [ ]:
conclusion_md = f"""# Conclusion tecnica - E5 Evaluacion de seleccion de slices sin mascara

## Objetivo

Se evaluo el impacto de reemplazar la seleccion de slice basada en mascara por estrategias que no usan mascara, cargando el modelo binario mejorado sin reentrenar.

## Motivacion

La referencia previa seleccionaba el slice sagital con mayor area de mascara. Esto es util para validacion tecnica, pero no representa inferencia real porque la mascara no esta disponible antes de predecir.

## Configuracion

- Modelo: `{MODEL_PATH}`
- Holdout: `{HOLDOUT_SELECTED_CSV}`
- Casos evaluados: {report['evaluated_cases']}
- Eje sagital: {SAGITTAL_AXIS}
- Target size: {TARGET_SIZE}
- Threshold: {THRESHOLD}
- Estrategias: {', '.join(STRATEGIES)}

## Resultados por estrategia

{summary_df.to_markdown(index=False)}

## Comparacion contra referencia con mascara

{comparison_summary_df.to_markdown(index=False)}

## Interpretacion

- Dice referencia `mask_max_area`: {report['reference_mask_max_area_dice']:.4f}
- Mejor estrategia sin mascara: `{report['best_no_mask_strategy_by_dice']}`
- Dice mejor estrategia sin mascara: {report['best_no_mask_dice']:.4f}
- Caida absoluta de Dice: {report['dice_drop_absolute']:.4f}
- Caida relativa de Dice: {report['dice_drop_relative']:.4f}
- IoU referencia: {report['reference_mask_max_area_iou']:.4f}
- IoU mejor estrategia sin mascara: {report['best_no_mask_iou']:.4f}

## Limitaciones

- Se mantiene segmentacion binaria.
- Se evalua un modelo 2D sobre slices sagitales.
- `all_slices_best_prediction` y `top_k_prediction_mean` requieren evaluar multiples slices, por lo que son mas costosas.
- No se evalua multiclase, axial, nnU-Net ni backend.

## Recomendacion

{report['recommendation']}
"""

conclusion_path = DOCS_ROOT / "E5_slice_selection_eval_conclusion.md"
conclusion_path.write_text(conclusion_md, encoding="utf-8")

print(conclusion_md)
print("Conclusion Markdown:", conclusion_path)

## 13. Criterio de aceptacion

El notebook es correcto si carga el modelo mejorado sin reentrenar, evalua los mismos casos holdout, compara una estrategia con mascara contra estrategias sin mascara, exporta metricas por caso, resumen, figuras, JSON y Markdown, y concluye si conviene seleccion sin mascara o inferencia multi-slice.